In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import gc
import numpy as np

from typing import List, Tuple, Dict

In [ ]:
X = pd.read_parquet(r'data/train/train_main_features.parquet').drop('customer_id', axis=1)
y = pd.read_parquet(r'data/train/train_target.parquet').drop('customer_id', axis=1)

In [ ]:
samples = len(X)

In [ ]:
TARGET = 'target_1_1'
y = y[TARGET] # <-- БУДЕТ МЕНЯТЬСЯ 41 раз :))

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
del X, y
gc.collect()

In [ ]:
category = []
for col in X_train.columns:
    if col.startswith('ca'):
        category.append(col)
        X_train[col] = X_train[col].astype(str)
        X_valid[col] = X_valid[col].astype(str)

Подбор гиперпараметров

In [ ]:
from catboost import CatBoostClassifier, Pool
import optuna

In [ ]:
train = Pool(
    data=X_train, label=y_train,
    cat_features=category
)
valid = Pool(
    data=X_valid, label=y_valid,
    cat_features=category
)

In [ ]:
# del X_train, y_train, X_valid, y_valid
# gc.collect()

In [ ]:
'''
здесь мы находим лучшие гиперпараметры для текущего таргета
'''

In [ ]:
params0 = {
    'eval_metric': 'AUC',

    'verbose': 50,  # отчёт каждые N новых деревьев
    # 'allow_writing_files': False,   # чтоб без отчётов в папку catboost_info
    'use_best_model': True,
    'early_stopping_rounds': 50,    # если метрика не растёт 50 деревьев, пруним

    'task_type': 'CPU',
    'thread_count': 10  # кол-во используемых потоков
}

def objective(trial: optuna.Trial):

    params1 = {
        'iterations': 1_200,    # специально много, потому что ставка не на это
        'depth': trial.suggest_int('depth', 4, 7),
        'learning_rate': trial.suggest_float('learning_rate', .01, .3, log=True),
    }
    params2 = {
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1., 50., log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0., 3.),
        'subsample': trial.suggest_float('subsample', .5, 1.),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', .5, 1.),
    }
    params3 = {
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 1, 50),
        'random_strength': trial.suggest_int('random_strength', 1, 7),
    }
    
    model = CatBoostClassifier(
        **params0,
        **params1,
        **params2,
        **params3
    )

    model.fit(train, eval_set=valid)

    best_score = model.get_best_score()['validation']['AUC']

    # ДЛЯ ПРУНИНГА
    # длина AUC-списка = кол-ву деревьев (iterations) ЛИБО пока не обрубится
    scores = model.get_evals_result()['validation']['AUC']
    for ep, score in enumerate(scores, 1):

        trial.report(score, ep)

        if trial.should_prune():
            raise optuna.TrialPruned()
    
    # ДЛЯ OPTUNA
    # потом можно переделать чтобы рисовать кривые обучения
    return best_score

In [ ]:
study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=5, # кол-во честных попыток (без прнунинга)
        n_warmup_steps=20
    )
)
study.optimize(
    objective,
    n_trials=30,    # ограничение в 30попыток поиска параметров
    # timeout=60*60,  # ограничение в минутах
    gc_after_trial=True,
)

In [ ]:
best_params = study.best_params

In [ ]:
'''
здесь мы уже нашли лучшие гиперпараметры и мы снова воссоздадим ту модель для анализа важности признаков
Определим же их
'''

In [ ]:
# модель с лучшими гиперами
model = CatBoostClassifier(
    **best_params,
    verbose=False
)
model.fit(train, eval_set=valid)

In [ ]:
# нужно отфильтровать и оставить top-N признаков
feature_importance = model.get_feature_importance(prettified=True)

# НУЖНО ОПРЕДЕЛИТЬ ОПТИМАЛЬНОЕ КОЛ-ВО ВАЖНЫХ ПРИЗНАКОВ
N = 150
best_features = feature_importance['FeatureId'].head(N).tolist()

In [ ]:
X_train = X_train[best_features]

In [ ]:
category = [col for col in X_train.columns if col.startswith('ca')]

In [ ]:
'''
здесь мы уже нашли и лучшие гиперы и наиболее важные признаки
'''

In [ ]:
from sklearn.model_selection import StratifiedKFold


kf = StratifiedKFold(n_splits=5, shuffle=True)

# samples * 1 <- вектор предсказаний для каждой строчки (1, потому что мы обучаем тут одну определенную модель)
# потом простэкуем их все и получим samples * 41, что и будет являться обучащими данными для второй модели
preds_model = np.zeros(len(X_train))

for train_idx, valid_idx in kf.split(X_train, y_train):

    X_tr, y_tr   = X_train.iloc[train_idx], y_train.iloc[train_idx]
    X_val, y_val = X_train.iloc[valid_idx], y_train.iloc[valid_idx]

    train_fold = Pool(
        data=X_tr, label=y_tr,
        cat_features=category
    )
    valid_fold = Pool(
        data=X_val, label=y_val,
        cat_features=category
    )
    del X_tr, y_tr, X_val, y_val


    model = CatBoostClassifier(
        **best_params,
        verbose=False
    )
    model.fit(train_fold, valid_fold)
    preds_model[valid_idx] = model.predict_proba(valid_fold)[:,1]

    del model

In [ ]:
df = pd.read_parquet(r'meta_data.parquet')
df[TARGET] = 

In [ ]:
# '''
# после того как я нашел лучшие гиперы
# я создаю новую модель с ними

# потом относительно этой модели я вытаскиваю важности признаков

# потом я обучаю новую модель с лучшими гиперами и только по полезным признакам кросс валидацией, и кросс-валидация (вектор samples * 1) отправляется в запись в мета-данные (это для обучения след модели)

# а потом я снова обучаю модель с лучшими гиперами по лучшим признакам, но уже на ВСЕХ данных и делаю ее снимок и отправляю в паапку всех снимков модели (это для submit)
# '''

In [ ]:
'''
мы обучили модель для обучения следующей с помощью кросс-валидации И создали вектор предсказаний
(этот вектор нужно записать в общие данные) + ТАКЖЕ нужно передавать имена признаков, на которых она обучалась
'''

In [ ]:
'''
теперь нужно просто обучить модель с лушими гиперами и важными признаками на ВСЕХ данных и сделать снимок
(снимок будет загружаться при финальном тестировании, при создании submit относительно 2 модели)
'''

In [ ]:
model = CatBoostClassifier(
    **best_params,
    verbose=False
)
X_all_train = Pool(
    data=X_train, label=y_train,
    cat_features=category
)
model.fit(X_all_train)
model.save_model(f'shapshots/{TARGET}.cbm')


data = {
    'target': TARGET,
    'best_params': best_params,
    'best_features': best_features,
    'category': category
}
import json
with open(f'configs/{TARGET}.json', 'w', encoding='utf-8') as file:

    json.dump(data, file, indent=4)